In [ ]:
!pip install monai scikit-image

In [ ]:
import os, re, glob

INPUT_ROOT = "/kaggle/input"


def find_volume_label_pairs():
    """/kaggle/input 전체를 재귀 탐색해서 volume-N.nii(.gz) / segmentation-N.nii(.gz)를
    번호(N)로 매칭한다. 정확한 중간 경로(owner/slug 등)를 가정하지 않고 /kaggle/input
    바로 아래부터 전부 훑는다 — Kaggle 환경에 따라 /kaggle/input/<slug>/... 일 수도,
    /kaggle/input/datasets/<owner>/<slug>/... 일 수도 있어서(실제로 후자였음)."""
    volumes, labels = {}, {}
    for path in glob.glob(os.path.join(INPUT_ROOT, "**", "volume*.nii*"), recursive=True):
        m = re.search(r"(\d+)", os.path.basename(path))
        if m:
            volumes[int(m.group(1))] = path
    for path in glob.glob(os.path.join(INPUT_ROOT, "**", "segmentation*.nii*"), recursive=True):
        m = re.search(r"(\d+)", os.path.basename(path))
        if m:
            labels[int(m.group(1))] = path
    common_ids = sorted(set(volumes) & set(labels))
    if not common_ids:
        raise FileNotFoundError(
            "volume/segmentation 쌍을 찾지 못했다. 아래 진단 셀로 실제 폴더 구조를 확인할 것."
        )
    print(f"매칭된 환자 케이스 {len(common_ids)}건 발견")
    return [{"image": volumes[i], "label": labels[i]} for i in common_ids]


# 진단용: 매칭이 안 될 때 이 셀만 따로 실행해서 실제 구조 확인
try:
    _ = find_volume_label_pairs()
except FileNotFoundError as e:
    print(e)
    print("\n실제 폴더 구조:")
    os.system("find /kaggle/input -maxdepth 6")
    raise


In [ ]:
import torch
from monai.data import DataLoader, Dataset, decollate_batch
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.networks.nets import SegResNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, SpatialPadd, RandCropByPosNegLabeld,
    RandFlipd, RandShiftIntensityd, EnsureTyped, Activations, AsDiscrete,
)
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROI_SIZE = (96, 96, 96)
BATCH_SIZE = 2
EPOCHS = 30
LR = 1e-4
VAL_FRACTION = 0.1
print(f"Device: {DEVICE}")


def get_transforms(train=True):
    base = [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
        ScaleIntensityRanged(keys=["image"], a_min=-200, a_max=200, b_min=0.0, b_max=1.0, clip=True),
        CropForegroundd(keys=["image", "label"], source_key="image"),
    ]
    if train:
        base += [
            # 일부 환자는 크롭 후 Z축(depth)이 ROI_SIZE(96)보다 얇음(슬라이스 두께가
            # 커서 총 슬라이스 수가 적은 케이스) — RandCropByPosNegLabeld 전에
            # 부족한 만큼 패딩해서 크롭 실패를 방지
            SpatialPadd(keys=["image", "label"], spatial_size=ROI_SIZE, mode="constant"),
            RandCropByPosNegLabeld(
                keys=["image", "label"], label_key="label", spatial_size=ROI_SIZE,
                pos=1, neg=1, num_samples=2, image_key="image", image_threshold=0,
            ),
            RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
            RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
        ]
    base.append(EnsureTyped(keys=["image", "label"]))
    return Compose(base)


data_dicts = find_volume_label_pairs()
import random
random.seed(42)
random.shuffle(data_dicts)
n_val = max(1, int(len(data_dicts) * VAL_FRACTION))
train_files, val_files = data_dicts[n_val:], data_dicts[:n_val]
print(f"train {len(train_files)} / val {len(val_files)}")

train_loader = DataLoader(Dataset(data=train_files, transform=get_transforms(True)), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(Dataset(data=val_files, transform=get_transforms(False)), batch_size=1, num_workers=2)

seg_model = SegResNet(blocks_down=[1, 2, 2, 4], blocks_up=[1, 1, 1], init_filters=16, in_channels=1, out_channels=3).to(DEVICE)
# 종양(class 2) 가중치를 높여 recall 우선 학습 (배경:간:종양 = 1:1:3)
loss_fn = DiceLoss(to_onehot_y=True, softmax=True, weight=torch.tensor([1.0, 1.0, 3.0]).to(DEVICE))
optimizer = torch.optim.Adam(seg_model.parameters(), lr=LR)
dice_metric = DiceMetric(include_background=False, reduction="mean")
post_pred = Compose([Activations(softmax=True), AsDiscrete(argmax=True, to_onehot=3)])
post_label = Compose([AsDiscrete(to_onehot=3)])

import pickle

seg_history = {
    "epoch_loss": [],
    "val_dice": [],
    "case_recall": [],
    "case_fpr": [],
    "tumor_tp": [],
    "tumor_fn": [],
    "tumor_fp": [],
    "tumor_tn": [],
}
best_dice = 0.0
for epoch in range(EPOCHS):
    seg_model.train()
    epoch_loss = 0.0
    for batch in tqdm(train_loader, desc=f"epoch {epoch+1} train"):
        images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
        optimizer.zero_grad()
        outputs = seg_model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)

    seg_model.eval()
    dice_metric.reset()
    tumor_tp = tumor_fn = tumor_fp = tumor_tn = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"epoch {epoch+1} val"):
            images, labels = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
            outputs = sliding_window_inference(images, roi_size=ROI_SIZE, sw_batch_size=4, predictor=seg_model)
            outputs = [post_pred(i) for i in decollate_batch(outputs)]
            labels_ = [post_label(i) for i in decollate_batch(labels)]
            dice_metric(y_pred=outputs, y=labels_)
            pred_classes = torch.argmax(outputs[0], dim=0)
            label_classes = torch.argmax(labels_[0], dim=0)
            gt_has_tumor = bool((label_classes == 2).any())
            pred_has_tumor = bool((pred_classes == 2).any())
            if gt_has_tumor and pred_has_tumor:
                tumor_tp += 1
            elif gt_has_tumor and not pred_has_tumor:
                tumor_fn += 1
            elif pred_has_tumor:
                tumor_fp += 1
            else:
                tumor_tn += 1
    val_dice = dice_metric.aggregate().item()
    case_recall = tumor_tp / (tumor_tp + tumor_fn) if (tumor_tp + tumor_fn) > 0 else float("nan")
    case_fp_rate = tumor_fp / (tumor_fp + tumor_tn) if (tumor_fp + tumor_tn) > 0 else float("nan")
    print(f"[Epoch {epoch+1}/{EPOCHS}] loss={epoch_loss:.4f} val_dice={val_dice:.4f} | "
          f"case_recall={case_recall:.3f}(TP={tumor_tp},FN={tumor_fn}) case_FPR={case_fp_rate:.3f}(FP={tumor_fp},TN={tumor_tn})")
    seg_history["epoch_loss"].append(epoch_loss)
    seg_history["val_dice"].append(val_dice)
    seg_history["case_recall"].append(case_recall)
    seg_history["case_fpr"].append(case_fp_rate)
    seg_history["tumor_tp"].append(tumor_tp)
    seg_history["tumor_fn"].append(tumor_fn)
    seg_history["tumor_fp"].append(tumor_fp)
    seg_history["tumor_tn"].append(tumor_tn)
    with open("ct_liver_seg_history.pkl", "wb") as f:
        pickle.dump(seg_history, f)
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(seg_model.state_dict(), "ct_liver_seg_best.pth")
        print(f"  -> 최고 성능 갱신, ct_liver_seg_best.pth 저장 (dice={best_dice:.4f})")

print(f"\n최종 최고 Dice(간+종양 평균): {best_dice:.4f}")
seg_history["best_dice"] = best_dice
with open("ct_liver_seg_history.pkl", "wb") as f:
    pickle.dump(seg_history, f)
print("ct_liver_seg_history.pkl 저장 완료")


In [ ]:
import numpy as np
from skimage import measure
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from monai.inferers import sliding_window_inference

LIVER_CHANNEL = 1  # 0=배경, 1=간, 2=종양

recon_model = SegResNet(blocks_down=[1, 2, 2, 4], blocks_up=[1, 1, 1], init_filters=16, in_channels=1, out_channels=3).to(DEVICE)
import os, glob
model_path = "ct_liver_seg_best.pth"
if not os.path.exists(model_path):
    matches = glob.glob("/kaggle/input/**/ct_liver_seg_best.pth", recursive=True)
    if matches:
        model_path = matches[0]
print(f"가중치 파일 로드: {model_path}")
recon_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
recon_model.eval()

sample = next(iter(val_loader))
post_pred_argmax = Compose([Activations(softmax=True), AsDiscrete(argmax=True)])
with torch.no_grad():
    out = sliding_window_inference(sample["image"].to(DEVICE), roi_size=(96, 96, 96), sw_batch_size=2, predictor=recon_model)
    mask_volume = post_pred_argmax(out[0]).cpu().numpy()[0]  # (D, H, W), 0/1/2

binary_volume = (mask_volume == LIVER_CHANNEL).astype(np.float32)
verts, faces, normals, values = measure.marching_cubes(binary_volume, level=0.5, spacing=(1.5, 1.5, 2.0))
print(f"복원된 3D 표면: 정점 {len(verts)}개, 삼각형면 {len(faces)}개")

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")
mesh = Poly3DCollection(verts[faces], alpha=0.7)
mesh.set_facecolor([0.7, 0.3, 0.3])
ax.add_collection3d(mesh)
ax.set_xlim(verts[:, 0].min(), verts[:, 0].max())
ax.set_ylim(verts[:, 1].min(), verts[:, 1].max())
ax.set_zlim(verts[:, 2].min(), verts[:, 2].max())
ax.set_title("Liver 3D Reconstruction (Marching Cubes)")
plt.tight_layout()
plt.savefig("liver_3d_reconstruction.png", dpi=150)
print("liver_3d_reconstruction.png 저장 완료")
plt.show()


In [ ]:
import os, re, glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader as TorchDataLoader
import nibabel as nib
from monai.networks.nets import UNet
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'data_dicts' not in globals():
    def find_volume_label_pairs():
        volumes, labels = {}, {}
        for p in glob.glob(os.path.join("/kaggle/input", "**", "volume*.nii*"), recursive=True):
            m = re.search(r"(\d+)", os.path.basename(p))
            if m: volumes[int(m.group(1))] = p
        for p in glob.glob(os.path.join("/kaggle/input", "**", "segmentation*.nii*"), recursive=True):
            m = re.search(r"(\d+)", os.path.basename(p))
            if m: labels[int(m.group(1))] = p
        common_ids = sorted(set(volumes) & set(labels))
        return [{"image": volumes[i], "label": labels[i]} for i in common_ids]
    data_dicts = find_volume_label_pairs()

DN_BATCH_SIZE = 16
DN_EPOCHS = 30
DN_LR = 1e-4
NOISE_STD = 0.15


def add_synthetic_low_dose_noise(image, noise_std=NOISE_STD):
    poisson_noise = np.random.poisson(image * 50) / 50.0 - image
    gaussian_noise = np.random.normal(0, noise_std, image.shape)
    noisy = image + poisson_noise * 0.5 + gaussian_noise
    return np.clip(noisy, 0, 1)


class CTDenoiseDataset(TorchDataset):
    def __init__(self, nii_paths, max_slices_per_vol=100):
        import cv2
        self.slices = []
        for path in nii_paths:
            vol = nib.load(path).get_fdata()
            vol = np.clip(vol, -200, 200)
            vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-8)
            valid_indices = [i for i in range(vol.shape[2]) if vol[:, :, i].std() > 0.01]
            if len(valid_indices) > max_slices_per_vol:
                step = len(valid_indices) // max_slices_per_vol
                valid_indices = valid_indices[::step][:max_slices_per_vol]
            for i in valid_indices:
                sl = vol[:, :, i]
                sl_128 = cv2.resize(sl.astype(np.float32), (128, 128))
                self.slices.append(sl_128)
        print(f"총 {len(self.slices)}개 슬라이스 로드 (메모리 최적화 완료)")

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        clean = self.slices[idx]
        noisy = add_synthetic_low_dose_noise(clean)
        return torch.tensor(noisy).unsqueeze(0).float(), torch.tensor(clean).unsqueeze(0).float()


def psnr(pred, target):
    mse = torch.mean((pred - target) ** 2)
    if mse == 0:
        return 100.0
    return 20 * torch.log10(1.0 / torch.sqrt(mse)).item()


nii_paths = [d["image"] for d in data_dicts]
print(f"CT 볼륨 {len(nii_paths)}개 발견")
n_val_dn = max(1, len(nii_paths) // 10)
train_paths, val_paths = nii_paths[n_val_dn:], nii_paths[:n_val_dn]

dn_train_loader = TorchDataLoader(CTDenoiseDataset(train_paths), batch_size=DN_BATCH_SIZE, shuffle=True, num_workers=2)
dn_val_loader = TorchDataLoader(CTDenoiseDataset(val_paths), batch_size=DN_BATCH_SIZE, num_workers=2)

dn_model = UNet(spatial_dims=2, in_channels=1, out_channels=1, channels=(16, 32, 64, 128), strides=(2, 2, 2)).to(DEVICE)
criterion = nn.MSELoss()
dn_optimizer = torch.optim.Adam(dn_model.parameters(), lr=DN_LR)

import pickle

dn_history = {
    "train_loss": [],
    "val_psnr": [],
}
best_psnr = 0.0
for epoch in range(DN_EPOCHS):
    dn_model.train()
    train_loss = 0.0
    for noisy, clean in tqdm(dn_train_loader, desc=f"epoch {epoch+1} train"):
        noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
        dn_optimizer.zero_grad()
        output = dn_model(noisy)
        loss = criterion(output, clean)
        loss.backward()
        dn_optimizer.step()
        train_loss += loss.item()
    train_loss /= len(dn_train_loader)

    dn_model.eval()
    val_psnr = 0.0
    with torch.no_grad():
        for noisy, clean in tqdm(dn_val_loader, desc=f"epoch {epoch+1} val"):
            noisy, clean = noisy.to(DEVICE), clean.to(DEVICE)
            output = dn_model(noisy).clamp(0, 1)
            val_psnr += psnr(output, clean)
    val_psnr /= len(dn_val_loader)
    print(f"[Epoch {epoch+1}/{DN_EPOCHS}] train_loss={train_loss:.4f} val_PSNR={val_psnr:.2f}dB")
    dn_history["train_loss"].append(train_loss)
    dn_history["val_psnr"].append(val_psnr)
    with open("ct_denoise_history.pkl", "wb") as f:
        pickle.dump(dn_history, f)
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(dn_model.state_dict(), "ct_denoise_best.pth")
        print(f"  -> 최고 성능 갱신, ct_denoise_best.pth 저장 (PSNR={best_psnr:.2f}dB)")

dn_model.eval()
noisy, clean = next(iter(dn_val_loader))
with torch.no_grad():
    denoised = dn_model(noisy.to(DEVICE)).clamp(0, 1).cpu()
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(noisy[0, 0], cmap="gray"); axes[0].set_title("저선량(노이즈)")
axes[1].imshow(denoised[0, 0], cmap="gray"); axes[1].set_title("AI 복원 결과")
axes[2].imshow(clean[0, 0], cmap="gray"); axes[2].set_title("원본(정답)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig("ct_denoise_comparison.png", dpi=150)
print(f"\n최종 최고 PSNR: {best_psnr:.2f}dB")
dn_history["best_psnr"] = best_psnr
with open("ct_denoise_history.pkl", "wb") as f:
    pickle.dump(dn_history, f)
print("ct_denoise_history.pkl 저장 완료")
print("ct_denoise_comparison.png 저장 완료")
